In [1]:
import importlib

#importlib.reload(test_spf)

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from galois import GF2
import tableau as ta

from gspf_ilp import generalised_spf_logical
from spf_graphs import crazy_graph, sample_lost_nodes
from decoder_methods import create_graph_code

In [ ]:
g = crazy_graph(4,4)
g = nx.to_numpy_array(g, dtype = np.uint16)
xlogi, zlogi, stabi = create_graph_code(g)
T = GF2(stabi)
numq = T.shape[1]//2
gg = 1
previous_meas = []
previous_meas = [ta.paulistring2tableau(ele, numq) for ele in previous_meas]

target = numq - 1          # output qubit O in tableau columns

# --- cache setup ---
cache = []                 # list of frozenset(support columns), full-tableau frame
max_cache = 1000

def pair_support(x, z, num_qubits, target_qubit):
    """Measured qubits = where x OR z is non-identity, excluding targer_qubit."""
    x = np.asarray(x).ravel().astype(int)
    z = np.asarray(z).ravel().astype(int)
    supp = set()
    for q in range(num_qubits):
        if q == target_qubit:
            continue
        if x[q] or x[q+num_qubits] or z[q] or z[q+num_qubits]:
            supp.add(q)
    return frozenset(supp)
# -------------------

fail_list = []
fit = []
num_shots = 5_00

hits = 0
solves = 0

lost_prob = np.linspace(0.1, 0.95, 10)
for p in lost_prob:
    print(p)
    fail = 0

    for ele in range(num_shots):
        lq = sample_lost_nodes(g, p)
        lost_cols = [node - 1 for node in lq]
        lost_set = set(int(c) for c in lost_cols)

        # --- cache lookup: any stored pattern that avoids all lost qubits? ---
        hit = False
        for supp in cache:
            if lost_set.isdisjoint(supp):
                hit = True
                break

        if hit:
            hits += 1
            fail += 1                      # success (a valid pattern survives)
            continue
        # -------------------------------------------------------------------

        solves += 1
        res = generalised_spf_logical(stabi, previous_meas, lost_cols, gg,
                                      target_qubit=target)
        if res["success"]:
            fail += 1
            # --- cache the found pattern (full-width x, z) ---
            supp = pair_support(res["x"], res["z"], numq, target)
            if supp not in cache:
                cache.append(supp)
                if len(cache) > max_cache:
                    cache.pop(0)           # FIFO eviction
            # ------------------------------------------------

    fail_list.append(fail/num_shots)
    fit.append((1-p**4)**4)

print(f"cache hits: {hits}, solves: {solves}, hit rate: {hits/(hits+solves):.3f}")

fail_list = np.asarray(fail_list)
yerr = np.sqrt(fail_list * (1 - fail_list) / num_shots)

plt.figure()
plt.title("Threshold plot for crazy graph")
plt.errorbar(lost_prob, fail_list, yerr=yerr, fmt="o", label="g-SPF")
plt.plot(lost_prob, fit, label=r"Fit $(1-p^{4})^{4}$")
plt.legend()
plt.show()

0.1
0.19444444444444445
oh no
oh no
0.28888888888888886
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
0.3833333333333333
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
0.47777777777777775
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
oh no
